In [6]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import csv

In [7]:
async def launch_browser(headless=False):
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=headless, slow_mo=250)  # slow_mo helps debugging
    page = await browser.new_page()
    return pw, browser, page

In [17]:
async def search_card(page, card_name, set_name):
    query = f"{card_name} {set_name}"
    await page.goto("https://www.tcgplayer.com", wait_until="domcontentloaded")

    # Wait for the search input
    await page.wait_for_selector("input#autocomplete-input")

    # Type query
    await page.fill("input#autocomplete-input", query)
    await page.keyboard.press("Enter")

    # --- Abrir Product Type ---
    await page.wait_for_selector("button[data-testid='filterBar-Product Type']")
    await page.click("button[data-testid='filterBar-Product Type']")

    # --- Seleccionar checkbox "Cards" ---
    # Ajusta el selector según el id/label real del checkbox
    await page.wait_for_selector("label[for='hfb-ProductType-Cards-filter']")
    await page.click("label[for='hfb-ProductType-Cards-filter']")

    # Wait for product cards to appear
    await page.wait_for_selector("section.product-card__product")

    # Get all cards, click the first one
    cards = await page.query_selector_all("section.product-card__product")
    if cards:
        await cards[0].click()
        await page.wait_for_timeout(3000)  # give it time to load product page
    else:
        print(f"No results found for {query}")

In [14]:
pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=False, slow_mo=250)
page = await browser.new_page()


In [8]:
await search_card(page, "Alakazam", "Base Set")

In [4]:
async def scrape_card_data(page, card_name, set_name, number_in_set, image_url):
    condition_list = ['Damaged', 'Heavily Played', 'Moderately Played', 'Lightly Played', 'Near Mint']
    results = []

    # --- Buscar carta ---
    await search_card(page, card_name, set_name)
    await page.reload()

    await page.wait_for_timeout(3000)

    # --- Abrir Printing popover ---
    await page.wait_for_selector("button[data-testid='filterBar-Printing']")
    await page.click("button[data-testid='filterBar-Printing']")
    await page.wait_for_selector("div[data-testid='searchFilterPrinting']")

    # --- Extraer opciones de Printing ---
    printing_checkboxes = await page.query_selector_all(
        "div[data-testid='searchFilterPrinting'] input.tcg-input-checkbox__input"
    )
    printing_options = {}
    for checkbox in printing_checkboxes:
        id_attr = await checkbox.get_attribute("id")
        label_text_el = await checkbox.evaluate_handle(
            "el => el.closest('label').querySelector('.tcg-input-checkbox__label-text')"
        )
        label_text = await label_text_el.inner_text() if label_text_el else id_attr
        printing_options[label_text] = id_attr

    # --- Abrir Condition popover ---
    await page.wait_for_selector("button[data-testid='filterBar-Condition']")
    await page.click("button[data-testid='filterBar-Condition']")
    await page.wait_for_selector("div[data-testid='searchFilterCondition']")

    # --- Extraer opciones de Condition ---
    condition_checkboxes = await page.query_selector_all(
        "div[data-testid='searchFilterCondition'] input.tcg-input-checkbox__input"
    )
    condition_options = {}
    for checkbox in condition_checkboxes:
        id_attr = await checkbox.get_attribute("id")
        label_text_el = await checkbox.evaluate_handle(
            "el => el.closest('label').querySelector('.tcg-input-checkbox__label-text')"
        )
        label_text = await label_text_el.inner_text() if label_text_el else id_attr
        condition_options[label_text] = id_attr

    # --- Iterar Printing × Condition ---
    for print_label, print_id in printing_options.items():
        for cond_label in condition_list:
            
            await page.click("button#clearFilters")
            await page.wait_for_timeout(200)

            # Idioma fijo en inglés
            await page.click("button[data-testid='filterBar-Language']")
            await page.click("label[for='hfb-Language-English-filter']")

            # Seleccionar impresión
            await page.click("button[data-testid='filterBar-Printing']")
            await page.click(f"label[for='hfb-{print_id}']")

            price_value = None

            # Seleccionar condición solo si existe
            if cond_label in condition_options:
                cond_id = condition_options[cond_label]
                await page.click("button[data-testid='filterBar-Condition']")
                await page.click(f"label[for='hfb-{cond_id}']")

                try:
                    await page.wait_for_selector("td .price-points__upper__price", timeout=5000)
                    price_text = await page.inner_text("td .price-points__upper__price")
                    price_value = float(price_text.replace("$", "").replace(",", ""))
                except:
                    price_value = None  # si no hay precio

            # --- Guardar resultado como una fila ---
            results.append({
                "card_name": card_name,
                "set_name": set_name,
                "number_in_set": number_in_set,
                "printing_option": print_label,
                "condition": cond_label,
                "market_price": price_value,
                "image": image_url
            })

    return results

In [36]:
# Abrir el popover de Language
await page.click("button[data-testid='filterBar-Language']")

# Esperar a que se renderice
await page.wait_for_selector("div[data-testid='searchFilterLanguage']")

# Click en el label de English
await page.click("label[for='hfb-Language-English-filter']")

# Espera a que la página actualice después del filtro
await page.wait_for_timeout(300)

In [21]:
# Click the Printing button to open the popover
await page.click("button[data-testid='filterBar-Printing']")

await page.wait_for_selector("label[for='hfb-Printing-Holofoil-filter']")
await page.click("label[for='hfb-Printing-Holofoil-filter']")
await page.wait_for_timeout(300)

# Espera a que la página actualice después de filtrar
await page.wait_for_timeout(300)


In [29]:
# Wait for the Printing button to appear
await page.wait_for_selector("button[data-testid='filterBar-Printing']")

# Click the Printing button to open the popover
await page.click("button[data-testid='filterBar-Printing']")

# Wait for the printing popover to render
await page.wait_for_selector("div[data-testid='searchFilterPrinting']")

# Get all printing checkboxes
printing_checkboxes = await page.query_selector_all(
    "div[data-testid='searchFilterPrinting'] input.tcg-input-checkbox__input"
)

# Diccionario { label: id }
printing_options = {}
for checkbox in printing_checkboxes:
    id_attr = await checkbox.get_attribute("id")

    # Buscar el texto visible del label
    label_text_el = await checkbox.evaluate_handle(
        "el => el.closest('label').querySelector('.tcg-input-checkbox__label-text')"
    )
    label_text = await label_text_el.inner_text() if label_text_el else id_attr

    # Guardamos solo una vez cada label (sobrescribe duplicados)
    printing_options[label_text] = id_attr

print("Printing options:", printing_options)

# --- Iterar sobre las opciones y hacer click ---
for label, checkbox_id in printing_options.items():
    print(f"Clicking on printing option: {label} ({checkbox_id})")

    # Click en el label asociado al checkbox
    await page.click(f"label[for='hfb-{checkbox_id}']")

    # Pequeña pausa para que la página actualice el filtro
    await page.wait_for_timeout(200)

Printing options: {'Unlimited Holofoil': 'Printing-UnlimitedHolofoil-filter', '1st Edition Holofoil': 'Printing-1stEditionHolofoil-filter'}
Clicking on printing option: Unlimited Holofoil (Printing-UnlimitedHolofoil-filter)
Clicking on printing option: 1st Edition Holofoil (Printing-1stEditionHolofoil-filter)


In [26]:
# --- Abrir el filtro Condition ---
await page.wait_for_selector("button[data-testid='filterBar-Condition']")
await page.click("button[data-testid='filterBar-Condition']")

# Esperar a que el popover de Condition aparezca
await page.wait_for_selector("div[data-testid='searchFilterCondition']")  # Ajusta el data-testid si es distinto

# Obtener todos los checkboxes dentro del popover de Condition
condition_checkboxes = await page.query_selector_all(
    "div[data-testid='searchFilterCondition'] input.tcg-input-checkbox__input"
)

# Diccionario { label: id }
condition_options = {}
for checkbox in condition_checkboxes:
    id_attr = await checkbox.get_attribute("id")
    
    # Buscar el texto visible del label
    label_text_el = await checkbox.evaluate_handle(
        "el => el.closest('label').querySelector('.tcg-input-checkbox__label-text')"
    )
    label_text = await label_text_el.inner_text() if label_text_el else id_attr

    condition_options[label_text] = id_attr

print("Condition options:", condition_options)

# Iterar sobre las opciones y hacer click en el label asociado
for label, checkbox_id in condition_options.items():
    print(f"Clicking on condition option: {label} ({checkbox_id})")
    await page.click(f"label[for='hfb-{checkbox_id}']")
    await page.wait_for_timeout(200)

Condition options: {'Lightly Played': 'Condition-LightlyPlayed-filter', 'Moderately Played': 'Condition-ModeratelyPlayed-filter', 'Heavily Played': 'Condition-HeavilyPlayed-filter', 'Near Mint': 'Condition-NearMint-filter'}
Clicking on condition option: Lightly Played (Condition-LightlyPlayed-filter)
Clicking on condition option: Moderately Played (Condition-ModeratelyPlayed-filter)
Clicking on condition option: Heavily Played (Condition-HeavilyPlayed-filter)
Clicking on condition option: Near Mint (Condition-NearMint-filter)


In [41]:
async def scrape_card_data(page, card_name, set_name, number_in_set, image_url):
    condition_list = ['Damaged', 'Heavily Played', 'Moderately Played', 'Lightly Played', 'Near Mint']
    results = []

    # --- Buscar carta ---
    await search_card(page, card_name, set_name)
    await page.reload()

    await page.wait_for_timeout(3000)

    # --- Abrir Printing popover ---
    await page.wait_for_selector("button[data-testid='filterBar-Printing']")
    await page.click("button[data-testid='filterBar-Printing']")
    await page.wait_for_selector("div[data-testid='searchFilterPrinting']")

    # --- Extraer opciones de Printing ---
    printing_checkboxes = await page.query_selector_all(
        "div[data-testid='searchFilterPrinting'] input.tcg-input-checkbox__input"
    )
    printing_options = {}
    for checkbox in printing_checkboxes:
        id_attr = await checkbox.get_attribute("id")
        label_text_el = await checkbox.evaluate_handle(
            "el => el.closest('label').querySelector('.tcg-input-checkbox__label-text')"
        )
        label_text = await label_text_el.inner_text() if label_text_el else id_attr
        printing_options[label_text] = id_attr

    # --- Abrir Condition popover ---
    await page.wait_for_selector("button[data-testid='filterBar-Condition']")
    await page.click("button[data-testid='filterBar-Condition']")
    await page.wait_for_selector("div[data-testid='searchFilterCondition']")

    # --- Extraer opciones de Condition ---
    condition_checkboxes = await page.query_selector_all(
        "div[data-testid='searchFilterCondition'] input.tcg-input-checkbox__input"
    )
    condition_options = {}
    for checkbox in condition_checkboxes:
        id_attr = await checkbox.get_attribute("id")
        label_text_el = await checkbox.evaluate_handle(
            "el => el.closest('label').querySelector('.tcg-input-checkbox__label-text')"
        )
        label_text = await label_text_el.inner_text() if label_text_el else id_attr
        condition_options[label_text] = id_attr

    # --- Iterar Printing × Condition ---
    for print_label, print_id in printing_options.items():
        for cond_label in condition_list:
            
            await page.click("button#clearFilters")
            await page.wait_for_timeout(200)

            # Idioma fijo en inglés
            await page.click("button[data-testid='filterBar-Language']")
            await page.click("label[for='hfb-Language-English-filter']")

            # Seleccionar impresión
            await page.click("button[data-testid='filterBar-Printing']")
            await page.click(f"label[for='hfb-{print_id}']")

            price_value = None

            # Seleccionar condición solo si existe
            if cond_label in condition_options:
                cond_id = condition_options[cond_label]
                await page.click("button[data-testid='filterBar-Condition']")
                await page.click(f"label[for='hfb-{cond_id}']")

                try:
                    await page.wait_for_selector("td .price-points__upper__price", timeout=5000)
                    price_text = await page.inner_text("td .price-points__upper__price")
                    price_value = float(price_text.replace("$", "").replace(",", ""))
                except:
                    price_value = None  # si no hay precio

            # --- Guardar resultado como una fila ---
            results.append({
                "card_name": card_name,
                "set_name": set_name,
                "number_in_set": number_in_set,
                "printing_option": print_label,
                "condition": cond_label,
                "market_price": price_value,
                "image": image_url
            })

    return results

In [18]:
results = await scrape_card_data(page, "Bulbasaur", "Detective Pikachu", 1, "https://images.pokemontcg.io/det1/1.png")
print(results)

[{'card_name': 'Bulbasaur', 'set_name': 'Detective Pikachu', 'number_in_set': 1, 'printing_option': 'Holofoil', 'condition': 'Damaged', 'market_price': 0.28, 'image': 'https://images.pokemontcg.io/det1/1.png'}, {'card_name': 'Bulbasaur', 'set_name': 'Detective Pikachu', 'number_in_set': 1, 'printing_option': 'Holofoil', 'condition': 'Heavily Played', 'market_price': 0.43, 'image': 'https://images.pokemontcg.io/det1/1.png'}, {'card_name': 'Bulbasaur', 'set_name': 'Detective Pikachu', 'number_in_set': 1, 'printing_option': 'Holofoil', 'condition': 'Moderately Played', 'market_price': 0.43, 'image': 'https://images.pokemontcg.io/det1/1.png'}, {'card_name': 'Bulbasaur', 'set_name': 'Detective Pikachu', 'number_in_set': 1, 'printing_option': 'Holofoil', 'condition': 'Lightly Played', 'market_price': 0.43, 'image': 'https://images.pokemontcg.io/det1/1.png'}, {'card_name': 'Bulbasaur', 'set_name': 'Detective Pikachu', 'number_in_set': 1, 'printing_option': 'Holofoil', 'condition': 'Near Mint'

In [44]:
# Guardar en un archivo CSV
with open("card_information.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("CSV generado: alakazam_results.csv")


CSV generado: alakazam_results.csv


In [30]:
# Espera a que aparezca el precio en la página
await page.wait_for_selector("td .price-points__upper__price")

# Extrae el texto del elemento
price_text = await page.inner_text("td .price-points__upper__price")

# Limpia el valor: quitar el símbolo "$" y convertir a float
price_value = float(price_text.replace("$", "").replace(",", ""))

print("Precio extraído:", price_value)

Precio extraído: 503.48


In [13]:
await browser.close()
await pw.stop()

In [ ]:
async def run_one_card(card_name, set_name, number_in_set, headless=False):
    pw, browser, page = await launch_browser(headless=headless)
    
    try:
        await search_card(page, card_name, set_name)
        data = await scrape_card_data(page, card_name, set_name, number_in_set)
    finally:
        await browser.close()
        await pw.stop()

    return data


In [ ]:
async def scrape_from_excel(file_path, headless=True):
    # Leer el Excel
    df = pd.read_excel(file_path)
    all_data = []

    # Iterar fila por fila
    for _, row in df.iterrows():
        card_name = row["CardName"]
        set_name = row["SetName"]
        number_in_set = row["NumberInSet"]
        image_url = row.get("ImageURL", None)  # si tienes columna de imagen

        # Scrape de cada carta
        data = await scrape_card_data(
            page=row["page"],  # si ya pasas la instancia de page por fila; si no, crearla antes
            card_name=card_name,
            set_name=set_name,
            number_in_set=number_in_set,
            image_url=image_url
        )
        all_data.extend(data)

    # Convertir a DataFrame
    result_df = pd.DataFrame(all_data)

    # Guardar a CSV
    result_df.to_csv("scrape_results.csv", index=False)
    print("CSV generado: scrape_results.csv")

    return result_df

# Usage:
# final_df = asyncio.run(scrape_from_excel("pokemon_cards.xlsx", headless=True))
# final_df.to_csv("pokemon_prices.csv", index=False)


In [3]:
df = pd.read_excel("lista_cartas.xlsx")

# Tomar solo la primera fila
first_row = df.iloc[0]

# Extraer valores
card_name = first_row["CardName"]
set_name = first_row["SetName"]
number_in_set = first_row["NumberInSet"]
image_url = first_row.get("ImageURL", None)  # opcional

# Imprimir para test
print("Card Name:", card_name)
print("Set Name:", set_name)
print("Number in Set:", number_in_set)
print("Image URL:", image_url)

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.